# ipynb/upset.ipynb - Upset 图 / Upset plot

## 项目背景 / Background
Upset 图
Upset plot

## 功能模块 / Modules
- 集合交集的 Upset 图
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: utils/fewshot_analysis_*.py
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [13]:
# ---------------------------------------------------------
# 1. 环境准备与数据加载
# ---------------------------------------------------------
if (!require("pacman")) install.packages("pacman")
# 建议加载 svglite 或 Cairo 包以增强导出兼容性
pacman::p_load(tidyverse, UpSetR, reticulate, Cairo)

# 定义莫兰迪色系
morandi_palette <- c("#91A3B0", "#B8AD9E", "#957E6E", "#A5A58D", "#6B705C")

# 12类修饰名称
mod_names <- c('Am', 'Atol', 'Cm', 'Gm', 'Tm', 'Y', 'ac4C', 'm1A', 'm5C', 'm6A', 'm6Am', 'm7G')

# 加载数据
np <- import("numpy")
data_path <- "/home/dc/vscode/vscode20260406/rgcnformer_sum/npy/human3/"
y_12class <- np$load(paste0(data_path, "12loc.npy"))

# 转换为 UpSet 格式的数据框
df_upset <- as.data.frame(y_12class)
colnames(df_upset) <- mod_names

# ---------------------------------------------------------
# 2. 统计并打印信息 (Intersection & Set Size)
# ---------------------------------------------------------
cat("\n[1/2] 正在统计每种修饰的 Set Size...\n")
set_sizes <- colSums(df_upset)
print(as.data.frame(set_sizes))

cat("\n[2/2] 正在统计各个 Intersection (交集组合) 的个数...\n")
intersections_df <- df_upset %>%
  group_by(across(everything())) %>%
  summarise(Count = n(), .groups = 'drop') %>%
  filter(rowSums(select(., all_of(mod_names))) > 0) %>% # 排除全0行
  arrange(desc(Count))

# 打印前 30 个交集组合 (可根据需要调整打印范围)
print(as.data.frame(intersections_df))

# ---------------------------------------------------------
# 3. 绘图逻辑
# ---------------------------------------------------------
plot_upset_func <- function() {
  # 计算频率 >= 50 的组合数量用于过滤
  patterns <- apply(df_upset, 1, paste, collapse = "")
  counts <- table(patterns)
  none_pattern <- paste(rep("0", ncol(df_upset)), collapse = "")
  counts <- counts[names(counts) != none_pattern]
  n_valid_intersections <- sum(counts >= 10)

  UpSetR::upset(df_upset, 
        sets = mod_names,
        nintersects = n_valid_intersections, 
        scale.intersections = "log10",      
        scale.sets = "log10",               
        set_size.show = TRUE,               
        number.angles = 45,                 
        main.bar.color = morandi_palette[1],
        sets.bar.color = morandi_palette[3],
        matrix.color = morandi_palette[5],
        shade.color = "#E0E0E0",
        order.by = "freq", 
        show.numbers = "yes",               
        text.scale = c(1.5, 1.5, 1.5, 1.5, 1.5, 1.5),
        point.size = 2.5, 
        line.size = 0.7)
}

# ---------------------------------------------------------
# 4. 保存文件 (优化 AI 可编辑性)
# ---------------------------------------------------------
# if(!dir.exists("png4")) dir.create("png4")

# 方案 A: 使用 cairo_pdf (最推荐，图层和字体更清晰)
Cairo::CairoPDF("/home/dc/vscode/vscode20260603/human_and_plant_few_shot/visualization/human/dataset/fig/upset_modifications_AI_ready.pdf", width = 32, height = 8)
plot_upset_func()
dev.off()

# 方案 B: 标准 PDF 设备 (需设置 useDingbats = FALSE)
# pdf("png4/upset_modifications.pdf", width = 16, height = 8, useDingbats = FALSE)
# plot_upset_func()
# dev.off()

# 保存预览用的图片
png("/home/dc/vscode/vscode20260603/human_and_plant_few_shot/visualization/human/dataset/fig/upset_modifications.png", width = 600, height = 800, res = 120)
plot_upset_func()
dev.off()

cat("\n所有文件已保存。PDF 已优化，可直接在 Adobe Illustrator 中编辑文字和形状。\n")


[1/2] 正在统计每种修饰的 Set Size...
     set_sizes
Am        1935
Atol     49435
Cm        2219
Gm        2196
Tm        2323
Y        36944
ac4C      3925
m1A      15489
m5C       3381
m6A     114195
m6Am      2221
m7G       1508

[2/2] 正在统计各个 Intersection (交集组合) 的个数...


   Am Atol Cm Gm Tm Y ac4C m1A m5C m6A m6Am m7G Count
1   0    0  0  0  0 0    0   0   0   1    0   0 84118
2   0    1  0  0  0 0    0   0   0   0    0   0 49435
3   0    0  0  0  0 1    0   0   0   1    0   0 28286
4   0    0  0  0  0 0    0   1   0   0    0   0 15489
5   0    0  0  0  0 1    0   0   0   0    0   0  7055
6   0    0  0  0  0 0    1   0   0   0    0   0  3925
7   0    0  0  0  0 0    0   0   1   0    0   0  3122
8   0    0  0  0  0 0    0   0   0   0    1   0  2221
9   0    0  0  0  1 0    0   0   0   0    0   0  2159
10  0    0  1  0  0 0    0   0   0   0    0   0  1811
11  1    0  0  0  0 0    0   0   0   0    0   0  1536
12  0    0  0  0  0 0    0   0   0   0    0   1  1508
13  0    0  0  1  0 0    0   0   0   0    0   0  1428
14  0    0  0  1  0 1    0   0   0   1    0   0   589
15  0    0  1  0  0 1    0   0   0   1    0   0   299
16  1    0  0  0  0 1    0   0   0   1    0   0   291
17  0    0  0  0  0 1    0   0   1   1    0   0   132
18  0    0  0  0  1 1    0  

pdf 
  2

pdf 
  2


所有文件已保存。PDF 已优化，可直接在 Adobe Illustrator 中编辑文字和形状。
